# 02_RAG与检索

**来源:**
- [LangChain Docs — Retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval)
- [LangChain Docs — RAG](https://docs.langchain.com/oss/python/deepagents/rag)
- [LangChain Docs — Knowledge Base](https://docs.langchain.com/oss/python/deepagents/knowledge-base)
- [LangChain Docs — Vector Stores](https://docs.langchain.com/oss/python/integrations/vectorstores)
- [LangChain Retrieval 概念](https://docs.langchain.com/oss/python/concepts/retrieval)

本 Notebook 全面讲解 RAG（检索增强生成）流程、各种检索器类型、向量存储及知识库构建。

## 目录

1. [RAG 流程概述](#1-rag-流程概述)
2. [文档加载 (Document Loaders)](#2-文档加载-document-loaders)
3. [文档分割 (Text Splitters)](#3-文档分割-text-splitters)
4. [向量存储 (Vector Stores)](#4-向量存储-vector-stores)
5. [检索器类型 (Retriever Types)](#5-检索器类型-retriever-types)
6. [知识库构建](#6-知识库构建)
7. [RAG 链式调用](#7-rag-链式调用)
8. [高级 RAG 技术](#8-高级-rag-技术)

**参考链接**
- [LangChain Retrieval 文档](https://docs.langchain.com/oss/python/deepagents/retrieval)
- [LangChain Vector Store 集成](https://docs.langchain.com/oss/python/integrations/vectorstores)
- [Chroma 文档](https://docs.trychroma.com/)
- [LangChain 文档加载器](https://python.langchain.com/docs/how_to/#document-loaders)

## 1. RAG 流程概述

RAG (Retrieval-Augmented Generation) 是一种将**检索**与**生成**相结合的技术范式：

```text
用户查询
    │
    ▼
[检索器] ←── [向量存储 / 搜索引擎 / 数据库]
    │
    ▼
相关文档片段
    │
    ▼
[LLM] ── 系统提示 + 检索结果 + 用户查询
    │
    ▼
生成回答
```

### 核心优势

| 优势 | 说明 |
|------|------|
| **知识更新** | 无需重新训练模型即可更新知识 |
| **可追溯** | 回答引用来源，增强可信度 |
| **降低成本** | 减少幻觉，减少 Prompt Token 消耗 |
| **领域定制** | 轻松适配企业内部文档、私有知识库 |

### RAG 三阶段

| 阶段 | 说明 | 组件 |
|------|------|------|
| **索引** | 准备知识库 | 文档加载器 → 文本分割器 → 嵌入模型 → 向量存储 |
| **检索** | 根据查询找到相关文档 | 检索器（向量搜索/关键词搜索/混合搜索） |
| **生成** | 结合检索结果生成回答 | LLM + 提示模板 |

## 2. 文档加载 (Document Loaders)

文档加载器将各种格式的源数据加载为 LangChain `Document` 对象。

### 支持的文档格式

| 格式 | 加载器 | 安装 |
|------|--------|------|
| PDF | `PyPDFLoader` | `pypdf` |
| Markdown | `TextLoader` | 内置 |
| HTML | `HTMLLoader` | `beautifulsoup4` |
| CSV | `CSVLoader` | 内置 |
| JSON | `JSONLoader` | 内置 |
| Word | `Docx2txtLoader` | `docx2txt` |
| PowerPoint | `UnstructuredPowerPointLoader` | `unstructured` |
| 网页 | `WebBaseLoader` | `beautifulsoup4` |
| 目录 | `DirectoryLoader` | 相应子加载器 |

In [ ]:
# pip install pypdf beautifulsoup4 chromadb

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    WebBaseLoader,
    DirectoryLoader,
)

# 加载 PDF
# loader = PyPDFLoader("document.pdf")
# docs = loader.load()

# 加载 Markdown
# loader = TextLoader("readme.md")
# docs = loader.load()

# 加载网页
# loader = WebBaseLoader("https://example.com")
# docs = loader.load()

# 批量加载目录
# loader = DirectoryLoader("./docs", glob="**/*.md")
# docs = loader.load()

print("文档加载器示例 (取消注释后运行)")

## 3. 文档分割 (Text Splitters)

将长文档分割成适合检索的块（chunks）。

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
)

# 推荐：递归字符文本分割器（按段落→句→词递归分割）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # 每块最多字符数
    chunk_overlap=200,     # 块之间重叠字符数
    separators=["\n\n", "\n", " ", ""],  # 分割优先级
)

# 按 Token 数分割（更精确控制上下文窗口）
token_splitter = TokenTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

# 按 Markdown 标题结构分割（保留文档结构）
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
)

print("文本分割器已定义")

# 使用示例
# splits = text_splitter.split_documents(docs)
# print(f"分割后文档数: {len(splits)}")

### 分割策略对比

| 分割器 | 适用场景 | 优点 |
|--------|---------|------|
| `RecursiveCharacterTextSplitter` | 通用文本 | 保持段落和句子完整性 |
| `TokenTextSplitter` | LLM Token 计数 | 精确控制 Token 用量 |
| `MarkdownHeaderTextSplitter` | Markdown 文档 | 保留标题层级结构 |
| `CharacterTextSplitter` | 简单场景 | 最轻量 |
| `SemanticChunker` | 语义分割 | 按语义边界分割（较慢） |

## 4. 向量存储 (Vector Stores)

向量存储是 RAG 的核心组件，用于存储文档嵌入向量并支持相似度搜索。

In [ ]:
from langchain_openai import OpenAIEmbeddings  # 或从其他 Provider
from langchain_chroma import Chroma

# 初始化嵌入模型
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    # openai_api_key="...",
)

# 创建向量存储（从文档）
# vector_store = Chroma.from_documents(
#     documents=splits,           # 分割后的文档列表
#     embedding=embeddings,       # 嵌入模型
#     persist_directory="./chroma_db",  # 持久化目录
# )

# 或者加载已有库
# vector_store = Chroma(
#     embedding_function=embeddings,
#     persist_directory="./chroma_db",
# )

print("Chroma 向量存储示例 (取消注释后运行)")

### 支持的向量存储

| 向量存储 | 安装 | 特点 |
|----------|------|------|
| **Chroma** | `chromadb` | 轻量、本地、开源 |
| **FAISS** | `faiss-cpu` | 高效本地搜索、Facebook |
| **Pinecone** | `langchain-pinecone` | 云原生、可扩展、托管 |
| **Weaviate** | `langchain-weaviate` | 云原生、混合搜索 |
| **Qdrant** | `langchain-qdrant` | 高性能、Rust 引擎 |
| **Milvus** | `langchain-milvus` | 分布式、企业级 |
| **PGVector** | `langchain-postgres` | PostgreSQL 扩展 |
| **Elasticsearch** | `langchain-elasticsearch` | 全文搜索 + 向量 |
| **Redis** | `langchain-redis` | 内存数据库、低延迟 |
| **LanceDB** | `lancedb` | 列式存储、嵌入式 |

### 嵌入模型选择

| 模型 | Provider | 维度 | 特点 |
|------|----------|------|------|
| `text-embedding-3-small` | OpenAI | 1536 | 性价比高 |
| `text-embedding-3-large` | OpenAI | 3072 | 精度高 |
| `gecko@003` | Google | 768 | Google 生态 |
| `cohere-embed-english-v3.0` | Cohere | 1024 | 企业级 |
| `all-MiniLM-L6-v2` | 本地(Sentence Transformers) | 384 | 免费、本地部署 |

## 5. 检索器类型 (Retriever Types)

检索器负责根据用户查询找到最相关的文档。

In [ ]:
from langchain.retrievers import (
    VectorStoreRetriever,       # 向量相似度搜索
    BM25Retriever,             # 关键词搜索
    EnsembleRetriever,          # 混合搜索
    ContextualCompressionRetriever,  # 上下文压缩
    MergerRetriever,            # 多检索器合并
)
from langchain.retrievers.document_compressors import (
    LLMChainExtractor,         # LLM 提取压缩
    EmbeddingsFilter,           # 嵌入过滤
)

# 1. 向量检索器（默认）
# retriever = vector_store.as_retriever(
#     search_type="similarity",       # similarity / mmr / similarity_score_threshold
#     search_kwargs={"k": 4},          # 返回前k个结果
# )

# 2. BM25 关键词检索器
# bm25_retriever = BM25Retriever.from_documents(splits)
# bm25_retriever.k = 4

# 3. 混合检索器（向量 + 关键词）
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[retriever, bm25_retriever],
#     weights=[0.7, 0.3],  # 向量搜索权重70%，关键词30%
# )

print("检索器类型示例 (取消注释后运行)")

### 检索器对比

| 检索器 | 搜索方式 | 优点 | 缺点 |
|--------|---------|------|------|
| `VectorStoreRetriever` | 语义搜索（向量相似度） | 理解语义、模糊匹配 | 需要嵌入模型、冷启动问题 |
| `BM25Retriever` | 关键词搜索（词频统计） | 无需训练、精确匹配 | 无法理解语义、同义词问题 |
| `EnsembleRetriever` | 混合（向量 + 关键词） | 兼顾语义和精确匹配 | 需要调权重 |
| `ContextualCompressionRetriever` | 检索后压缩 | 减少无关信息、提高精度 | 增加 LLM 调用开销 |
| `MergerRetriever` | 多源合并 | 整合多个检索源 | 结果可能冗余 |

### 搜索类型 (search_type)

| 类型 | 说明 |
|------|------|
| `similarity` | 简单相似度搜索，返回最相似的 k 个结果 |
| `mmr` | 最大边际相关性，增加结果多样性 |
| `similarity_score_threshold` | 阈值过滤，只返回得分高于阈值的结果 |

## 6. 知识库构建

构建一个可查询的知识库通常包含以下步骤：

In [ ]:
# 完整的知识库构建流程
import os
from pathlib import Path

def build_knowledge_base(
    docs_dir: str,
    persist_dir: str = "./chroma_db",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
):
    """构建知识库的完整流程"""
    from langchain_community.document_loaders import DirectoryLoader
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings
    from langchain_chroma import Chroma

    # 1. 加载文档
    print(f"从 {docs_dir} 加载文档...")
    loader = DirectoryLoader(docs_dir, glob="**/*.md")
    docs = loader.load()
    print(f"  加载了 {len(docs)} 个文档")

    # 2. 分割文档
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    splits = splitter.split_documents(docs)
    print(f"  分割为 {len(splits)} 个块")

    # 3. 创建嵌入并存储
    print("创建嵌入并存储到向量库...")
    embeddings = OpenAIEmbeddings()
    vector_store = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    print(f"  知识库已存储到 {persist_dir}")

    return vector_store


# 使用示例
# vector_store = build_knowledge_base("./my_docs")
# retriever = vector_store.as_retriever(search_kwargs={"k": 4})

### 6.1 Deep Agents 知识库集成

Deep Agents 提供了更高级的知识库集成方式，通过 `rag` 相关工具实现检索增强：

In [ ]:
# Deep Agents 知识库集成
from deepagents import create_deep_agent
from langchain.tools import tool

# 将检索器封装为 Agent 可用的工具
@tool
def search_knowledge_base(query: str) -> str:
    """从公司知识库中搜索相关信息。
    当你需要了解公司政策、产品文档或内部知识时使用。
    """
    # 实际代码中应连接向量存储
    # docs = retriever.invoke(query)
    # return "\n\n".join(doc.page_content for doc in docs)
    return f"关于 '{query}' 的知识库检索结果..."

# 创建带知识库工具的 Agent
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[search_knowledge_base],
    system_prompt="你是一个客服助手。当用户提问时，先从知识库检索再回答。",
)

## 7. RAG 链式调用

使用 LangChain 表达式语言（LCEL）构建完整的 RAG 链：

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI

# RAG 提示模板
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个基于检索增强生成的问答助手。
    请根据以下检索到的上下文回答问题。
    如果上下文中没有足够信息，请明确说明。
    
    上下文:
    {context}
    """),
    ("human", "{question}"),
])

# 格式化检索结果的函数
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL 链
# rag_chain = (
#     RunnableParallel(
#         {
#             "context": retriever | format_docs,  # 检索 + 格式化
#             "question": RunnablePassthrough(),    # 传递用户问题
#         }
#     )
#     | rag_prompt          # 填充提示模板
#     | ChatOpenAI(model="gpt-5.4")  # LLM 生成
# )

# 调用
# result = rag_chain.invoke("公司的年假政策是什么？")
# print(result.content)

print("RAG 链定义 (取消注释后运行)")

## 8. 高级 RAG 技术

### 8.1 多查询检索 (Multi-Query Retriever)

将用户查询扩展为多个相似查询，分别检索后合并结果：

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

# 多查询检索器
# multi_query_retriever = MultiQueryRetriever.from_llm(
#     retriever=retriever,
#     llm=ChatOpenAI(model="gpt-5.4"),
# )

# 父文档检索 (Parent Document Retriever)
# 检索小块后返回所在的大块，保留上下文
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# parent_retriever = ParentDocumentRetriever(
#     vectorstore=vector_store,
#     docstore=InMemoryStore(),
#     child_splitter=RecursiveCharacterTextSplitter(chunk_size=200),
#     parent_splitter=RecursiveCharacterTextSplitter(chunk_size=2000),
# )

print("高级 RAG 检索器示例")

### 8.2 技术对比表

| 技术 | 适用场景 | 复杂度 | 效果提升 |
|------|---------|--------|---------|
| 基础向量检索 | 通用场景 | 低 | 基准线 |
| 混合检索 (Ensemble) | 需要精确匹配+语义理解 | 中 | +10-20% |
| 多查询检索 | 用户查询模糊 | 中 | +15-25% |
| 父文档检索 | 需要完整上下文 | 中 | +10-15% |
| 上下文压缩 | Token 预算有限 | 高 | 减少 Token 消耗 |
| 重排序 (Rerank) | 需要高精度 | 高 | +20-30% |
| 自适应 RAG | 查询类型多样 | 高 | 综合最优 |

---

**参考链接**
- [LangChain Retrieval 文档](https://docs.langchain.com/oss/python/deepagents/retrieval)
- [LangChain RAG 文档](https://docs.langchain.com/oss/python/deepagents/rag)
- [LangChain 知识库文档](https://docs.langchain.com/oss/python/deepagents/knowledge-base)
- [LangChain 向量存储集成](https://docs.langchain.com/oss/python/integrations/vectorstores)
- [Chroma 文档](https://docs.trychroma.com/)
- [FAISS 文档](https://github.com/facebookresearch/faiss)
- [Pinecone 文档](https://www.pinecone.io/docs/)